In [27]:
import json
import os
import soundfile as sf
import torch
import numpy as np
from nnAudio.features import CQT2010v2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import sys
sys.path.append('..')
from FMEncoderChain import FMEncoderChain
from fm_chain import fm_renderer
from loss_batch import compute_spectrogram_cqt_batched, cqt_spectrogram_loss_batched

def load_nsynth_samples(nsynth_dir = "/home/marcus/Music/nsynth-valid/",
                        instrument_families=None,
                        max_samples=50,
                        midi_notes=[45]):
    json_path = os.path.join(nsynth_dir, 'examples.json')
    audio_dir = os.path.join(nsynth_dir, 'audio')

    with open(json_path, 'r') as f:
        examples = json.load(f)

    # filter for pitch
    sel_examples = {
        k: v for k, v in examples.items()
        if v['pitch'] in midi_notes
    }
    if instrument_families:
        sel_examples = {
            k: v for k, v in sel_examples.items()
            if v['instrument_family_str'] in instrument_families
        }

    print(f"Found {len(sel_examples)} samples")

    samples = []
    for note_str, meta in list(sel_examples.items())[:max_samples]:
        wav_path = os.path.join(audio_dir, f'{note_str}.wav')
        if not os.path.exists(wav_path):
            continue
        audio, sr = sf.read(wav_path)
        assert sr == 16000, f"expected 16khz, got {sr}"
        audio = torch.tensor(audio, dtype = torch.float32)
        samples.append({
            'note_str': note_str,
            'audio': audio,
            'family': meta['instrument_family_str'],
            'source': meta['instrument_source_str'],
            'qualities': meta['qualities_str'],
            'pitch':meta['pitch'],
        })

    return samples

def compute_sustain_spectrogram(audio, cqt_transform,
                                 Fs = 16000, hop_size = 512,
                                 t_start =0.2, t_end=2.0):
     frame_start = int(t_start * Fs / hop_size)
     frame_end = int(t_end * Fs / hop_size)

     with torch.no_grad():
         spec = cqt_transform(audio.unsqueeze(0)) # [1, n_bins, time_frames]
         spec = spec.abs()
         spec = torch.clamp(spec, min=0.0)
         spec = torch.log1p(spec)

         spec = spec[:,:, frame_start:frame_end]
         spec = spec.mean(dim=2)
         spec = spec.squeeze(0)
     
     return spec

QUALITY_NAMES = [
    'bright', 'dark', 'distortion', 'fast_decay', 'long_release',
    'multiphonic', 'nonlinear_env', 'percussive', 'reverb', 'tempo_synced'
]
 
FAMILY_COLORS = {
    'bass':       '#378ADD',
    'brass':      '#D85A30',
    'flute':      '#1D9E75',
    'guitar':     '#534AB7',
    'keyboard':   '#BA7517',
    'mallet':     '#D4537E',
    'organ':      '#639922',
    'reed':       '#E24B4A',
    'string':     '#0F6E56',
    'synth_lead': '#993556',
    'vocal':      '#888780',
}
 
 
def decode_qualities(quality_vec):
    """Convert binary quality vector to list of quality name strings."""
    if isinstance(quality_vec, list):
        return [QUALITY_NAMES[i] for i, v in enumerate(quality_vec) if v == 1]
    return []
 
 
def analyse_nsynth_results(results):
    """
    Full analysis of NSynth evaluation results.
 
    results: list of dicts with keys:
        idx, spectral_loss, family, source, qualities,
        gt_spec [n_bins], pred_spec [n_bins],
        pred_r0, pred_r1, pred_l0, pred_l1  (optional)
 
    Produces:
        - Best/worst spectral fits with spectrogram overlays
        - Per-family and per-source breakdown
        - Quality correlation analysis
        - Summary statistics table
    """
 
    # ── decode qualities if stored as binary vectors ──────────────────────────
    for r in results:
        if isinstance(r.get('qualities'), list) and len(r['qualities']) == 10:
            r['quality_names'] = decode_qualities(r['qualities'])
        else:
            r['quality_names'] = r.get('qualities', [])
 
    # ── sort ──────────────────────────────────────────────────────────────────
    by_spectral = sorted(results, key=lambda x: x['spectral_loss'])
    n_show = min(5, len(results))
 
    # ── figure layout ─────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 18))
    fig.suptitle('NSynth A2 — FMEncoderChain Evaluation', fontsize=14,
                 fontweight='bold', y=0.98)
 
    gs = gridspec.GridSpec(4, 5, figure=fig,
                           hspace=0.55, wspace=0.35,
                           left=0.06, right=0.97,
                           top=0.93, bottom=0.05)
 
    # ── row 0: summary stats ──────────────────────────────────────────────────
    ax_dist = fig.add_subplot(gs[0, 0])
    losses = [r['spectral_loss'] for r in results]
    ax_dist.hist(losses, bins=30, color='#378ADD', alpha=0.8, edgecolor='none')
    ax_dist.axvline(np.median(losses), color='#D85A30', linewidth=1.5,
                    linestyle='--', label=f'median {np.median(losses):.4f}')
    ax_dist.axvline(np.mean(losses), color='#1D9E75', linewidth=1.5,
                    linestyle=':', label=f'mean {np.mean(losses):.4f}')
    ax_dist.set_title('spectral loss distribution', fontsize=9)
    ax_dist.set_xlabel('spectral loss', fontsize=8)
    ax_dist.legend(fontsize=7)
 
    # ── per-family box plot ───────────────────────────────────────────────────
    ax_fam = fig.add_subplot(gs[0, 1:3])
    family_losses = defaultdict(list)
    for r in results:
        family_losses[r['family']].append(r['spectral_loss'])
    families = sorted(family_losses.keys(), key=lambda f: np.median(family_losses[f]))
    fam_data = [family_losses[f] for f in families]
    fam_colors = [FAMILY_COLORS.get(f, '#888780') for f in families]
    bp = ax_fam.boxplot(fam_data, patch_artist=True, vert=True,
                        medianprops={'color': 'white', 'linewidth': 1.5})
    for patch, color in zip(bp['boxes'], fam_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    ax_fam.set_xticks(range(1, len(families) + 1))
    ax_fam.set_xticklabels(families, rotation=35, ha='right', fontsize=7)
    ax_fam.set_title('spectral loss by instrument family', fontsize=9)
    ax_fam.set_ylabel('spectral loss', fontsize=8)
 
    # ── per-source bar ────────────────────────────────────────────────────────
    ax_src = fig.add_subplot(gs[0, 3])
    source_losses = defaultdict(list)
    for r in results:
        source_losses[r.get('source', r.get('sorce', 'unknown'))].append(
            r['spectral_loss'])
    src_names = list(source_losses.keys())
    src_means = [np.mean(source_losses[s]) for s in src_names]
    src_colors = ['#534AB7', '#D85A30', '#1D9E75']
    ax_src.bar(src_names, src_means,
               color=src_colors[:len(src_names)], alpha=0.8, edgecolor='none')
    ax_src.set_title('mean loss by source', fontsize=9)
    ax_src.set_ylabel('mean spectral loss', fontsize=8)
    ax_src.tick_params(axis='x', labelsize=8)
 
    # ── quality correlation ───────────────────────────────────────────────────
    ax_qual = fig.add_subplot(gs[0, 4])
    quality_losses = defaultdict(list)
    no_quality_losses = []
    for r in results:
        qnames = r['quality_names']
        if qnames:
            for q in qnames:
                quality_losses[q].append(r['spectral_loss'])
        else:
            no_quality_losses.append(r['spectral_loss'])
 
    if quality_losses:
        q_names_sorted = sorted(quality_losses.keys(),
                                key=lambda q: np.mean(quality_losses[q]))
        q_means = [np.mean(quality_losses[q]) for q in q_names_sorted]
        colors = ['#D85A30' if m > np.median(losses) else '#1D9E75'
                  for m in q_means]
        ax_qual.barh(q_names_sorted, q_means, color=colors, alpha=0.8,
                     edgecolor='none')
        ax_qual.axvline(np.median(losses), color='#378ADD', linewidth=1,
                        linestyle='--', label='overall median')
        ax_qual.set_title('mean loss by quality tag', fontsize=9)
        ax_qual.set_xlabel('mean spectral loss', fontsize=8)
        ax_qual.tick_params(axis='y', labelsize=7)
        ax_qual.legend(fontsize=7)
 
    # ── row 1: best spectral fits ─────────────────────────────────────────────
    for i, r in enumerate(by_spectral[:n_show]):
        ax = fig.add_subplot(gs[1, i])
        ax.plot(r['gt_spec'], color='#378ADD', linewidth=0.9, label='target')
        ax.plot(r['pred_spec'], color='#D85A30', linewidth=0.9,
                linestyle='--', label='pred')
        qstr = ', '.join(r['quality_names'][:2]) if r['quality_names'] else '—'
        ax.set_title(
            f"#{i+1} best  sl={r['spectral_loss']:.4f}\n"
            f"{r['family']} / {r.get('source', r.get('sorce','?'))}\n"
            f"{qstr}",
            fontsize=7
        )
        if i == 0:
            ax.legend(fontsize=6)
        ax.set_xticks([]); ax.set_yticks([])
 
    # ── row 2: worst spectral fits ────────────────────────────────────────────
    for i, r in enumerate(by_spectral[-n_show:]):
        ax = fig.add_subplot(gs[2, i])
        ax.plot(r['gt_spec'], color='#378ADD', linewidth=0.9, label='target')
        ax.plot(r['pred_spec'], color='#D85A30', linewidth=0.9,
                linestyle='--', label='pred')
        qstr = ', '.join(r['quality_names'][:2]) if r['quality_names'] else '—'
        ax.set_title(
            f"#{i+1} worst  sl={r['spectral_loss']:.4f}\n"
            f"{r['family']} / {r.get('source', r.get('sorce','?'))}\n"
            f"{qstr}",
            fontsize=7
        )
        ax.set_xticks([]); ax.set_yticks([])
 
    # ── row 3: family median ranking + summary table ──────────────────────────
    ax_rank = fig.add_subplot(gs[3, :3])
    fam_medians = {f: np.median(v) for f, v in family_losses.items()}
    fam_counts  = {f: len(v) for f, v in family_losses.items()}
    sorted_fams = sorted(fam_medians.keys(), key=lambda f: fam_medians[f])
    bar_colors  = [FAMILY_COLORS.get(f, '#888780') for f in sorted_fams]
    bars = ax_rank.barh(
        sorted_fams,
        [fam_medians[f] for f in sorted_fams],
        color=bar_colors, alpha=0.85, edgecolor='none'
    )
    for bar, f in zip(bars, sorted_fams):
        ax_rank.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
                     f'n={fam_counts[f]}', va='center', fontsize=7)
    ax_rank.set_title('median spectral loss by family (lower = better fit)',
                      fontsize=9)
    ax_rank.set_xlabel('median spectral loss', fontsize=8)
    ax_rank.tick_params(axis='y', labelsize=8)
 
    # ── summary text table ────────────────────────────────────────────────────
    ax_tbl = fig.add_subplot(gs[3, 3:])
    ax_tbl.axis('off')
    lines = [
        f"{'SUMMARY':^40}",
        f"{'─'*40}",
        f"  total samples:     {len(results)}",
        f"  mean spec loss:    {np.mean(losses):.4f}",
        f"  median spec loss:  {np.median(losses):.4f}",
        f"  std spec loss:     {np.std(losses):.4f}",
        f"  best:              {np.min(losses):.4f}",
        f"  worst:             {np.max(losses):.4f}",
        f"{'─'*40}",
        f"  best family:    {sorted_fams[0]}  ({fam_medians[sorted_fams[0]]:.4f})",
        f"  worst family:   {sorted_fams[-1]}  ({fam_medians[sorted_fams[-1]]:.4f})",
    ]
    if quality_losses:
        best_q  = min(quality_losses, key=lambda q: np.mean(quality_losses[q]))
        worst_q = max(quality_losses, key=lambda q: np.mean(quality_losses[q]))
        lines += [
            f"{'─'*40}",
            f"  best quality:   {best_q}  ({np.mean(quality_losses[best_q]):.4f})",
            f"  worst quality:  {worst_q}  ({np.mean(quality_losses[worst_q]):.4f})",
        ]
    ax_tbl.text(0.05, 0.95, '\n'.join(lines), transform=ax_tbl.transAxes,
                fontsize=8, verticalalignment='top',
                fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='none',
                          edgecolor='lightgray', alpha=0.5))
 
    plt.show()
 
 
def print_nsynth_table(results, n_show=10):
    """Print ranked table of best and worst NSynth fits."""
    by_spectral = sorted(results, key=lambda x: x['spectral_loss'])
 
    def row(r, rank):
        qstr = ','.join(r['quality_names'][:3]) if r['quality_names'] else '—'
        src  = r.get('source', r.get('sorce', '?'))
        return (f"{rank:>4}  {r['spectral_loss']:>10.4f}  "
                f"{r['family']:<12}  {src:<11}  {qstr}")
 
    header = f"{'rank':>4}  {'spec_loss':>10}  {'family':<12}  {'source':<11}  qualities"
    sep    = "─" * 70
 
    print(f"\n{'BEST NSynth fits':^70}")
    print(sep); print(header); print(sep)
    for i, r in enumerate(by_spectral[:n_show]):
        print(row(r, i+1))
 
    print(f"\n{'WORST NSynth fits':^70}")
    print(sep); print(header); print(sep)
    for i, r in enumerate(by_spectral[-n_show:]):
        print(row(r, len(results)-n_show+i+1))
 
    # per-family summary
    family_losses = defaultdict(list)
    for r in results:
        family_losses[r['family']].append(r['spectral_loss'])
    print(f"\n{'PER-FAMILY SUMMARY':^70}")
    print(sep)
    print(f"{'family':<14}  {'n':>4}  {'mean':>8}  {'median':>8}  {'std':>8}")
    print(sep)
    for fam in sorted(family_losses, key=lambda f: np.median(family_losses[f])):
        v = family_losses[fam]
        print(f"{fam:<14}  {len(v):>4}  {np.mean(v):>8.4f}  "
              f"{np.median(v):>8.4f}  {np.std(v):>8.4f}")

In [31]:
Fs = 16000
hop_size = 512
bins_per_octave = 32
n_octaves=7
n_bins = bins_per_octave*n_octaves
f0 = 110 #hz
notes = [45] # A2, 110hz midi note 45 

# load encoder
checkpointPath = '/home/marcus/FM_DDSP/chainEncoders/encoder_epoch_199.pt' # im sick of camel case
encoder = FMEncoderChain(n_bins=224)
encoder.load_state_dict(torch.load(checkpointPath))
encoder.eval()

cqt_transform = CQT2010v2(sr = Fs,
                          hop_length = hop_size,
                          n_bins = n_bins,
                          bins_per_octave = bins_per_octave)
samples = load_nsynth_samples()
#spec = compute_sustain_spectrogram(samples[1]['audio'], cqt_transform)
results = []
with torch.no_grad():
    for idx in range(len(samples)):
        spec = compute_sustain_spectrogram(cqt_transform,samples[idx]['audio'])
        spec = spec.float().unsqueeze(0)

        # encoder forward pass
        predicted = encoder(spec)

        # render predicted audio
        f0_t = torch.tensor([f0], dtype=torch.float32)
        audio_pred = fm_renderer(
            f0_t,
            predicted['ratios'],
            predicted['levels'],
            Fs, duration
        )

        # compute predicted spectrogram
        pred_spec = compute_spectrogram_cqt_batched(audio_pred, cqt_transform)

        # spectral loss
        spectral_loss = cqt_spectrogram_loss_batched(pred_spec, spec).item()

        results.append({
            'idx':idx,
            'spectral_loss' : spectral_loss,
            'family':samples[idx]['family'],
            'source':samples[idx]['source'],
            'qualites':samples[idx]['qualities'],
            'gt_spec': spec[0].cpu().numpy(),
            'pred_spec': pred_spec.cpu().numpy(),
            'pred_r0':      predicted['ratios'][0,0].item(),
            'pred_r1':      predicted['ratios'][0,1].item(),
            'pred_l0':      predicted['levels'][0,0].item(),
            'pred_l1':      predicted['levels'][0,1].item(),
        })

print_nsynth_table(results)
analyze_nsynth_results(results)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.